In [1]:
# -*- coding: utf-8 -*-
"""01_Download_Images_Improved.py
Robust image downloader with comprehensive error handling and retry logic
"""

import os
import sys
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from datetime import datetime

# ============================================================================
# 1. SETUP ENVIRONMENT & PATHS
# ============================================================================
print("="*70)
print("ROBUST IMAGE DOWNLOADER")
print("="*70)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Resolve base path
BASE_PATH = os.path.abspath('AMLC_2025')
DATA_PATH = os.path.join(BASE_PATH, 'data')
IMAGE_PATH = os.path.join(BASE_PATH, 'image')

print(f"Current Working Directory: {os.getcwd()}")
print(f"Base Path: {BASE_PATH}")
print(f"Data Path: {DATA_PATH}")
print(f"Image Path: {IMAGE_PATH}\n")

# Create directories
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(IMAGE_PATH, exist_ok=True)

# ============================================================================
# 2. LOAD DATASETS
# ============================================================================
print("Loading datasets...")

train_csv = os.path.join(DATA_PATH, 'train.csv')
test_csv = os.path.join(DATA_PATH, 'test.csv')

# Validate files exist
for file_path in [train_csv, test_csv]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Missing file: {file_path}")

try:
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    print(f"✅ Train dataset loaded: {len(train_df)} rows")
    print(f"✅ Test dataset loaded: {len(test_df)} rows")
except Exception as e:
    raise Exception(f"❌ Error loading CSVs: {e}")

# Validate image_link column exists
for name, df in [('train', train_df), ('test', test_df)]:
    if 'image_link' not in df.columns:
        raise KeyError(f"❌ {name}.csv missing 'image_link' column")

print()

# ============================================================================
# 3. SETUP OUTPUT FOLDERS
# ============================================================================
TRAIN_IMAGE_FULL_PATH = os.path.join(IMAGE_PATH, 'train_full')
TEST_IMAGE_FULL_PATH = os.path.join(IMAGE_PATH, 'test_full')

os.makedirs(TRAIN_IMAGE_FULL_PATH, exist_ok=True)
os.makedirs(TEST_IMAGE_FULL_PATH, exist_ok=True)

print(f"Train images folder: {TRAIN_IMAGE_FULL_PATH}")
print(f"Test images folder: {TEST_IMAGE_FULL_PATH}\n")

# ============================================================================
# 4. ROBUST IMAGE DOWNLOADER
# ============================================================================
class ImageDownloader:
    def __init__(self, max_workers=16, timeout=15, retries=3):
        self.max_workers = max_workers
        self.timeout = timeout
        self.retries = retries
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        self.stats = {
            'successful': 0,
            'failed': 0,
            'skipped': 0,
            'errors': []
        }
    
    def get_filename_from_url(self, url):
        """Extract filename from URL"""
        try:
            parsed_url = urlparse(url)
            filename = os.path.basename(parsed_url.path)
            if not filename or '.' not in filename:
                filename = f"image_{hash(url) % 10000}.jpg"
            return filename
        except Exception as e:
            return f"image_{hash(url) % 10000}.jpg"
    
    def download_image(self, url, output_folder):
        """Download a single image with retry logic"""
        if not url or pd.isna(url):
            self.stats['skipped'] += 1
            return False
        
        filename = self.get_filename_from_url(url)
        filepath = os.path.join(output_folder, filename)
        
        # Skip if already exists
        if os.path.exists(filepath):
            self.stats['skipped'] += 1
            return True
        
        # Retry logic
        for attempt in range(self.retries):
            try:
                response = self.session.get(url, timeout=self.timeout, stream=True)
                response.raise_for_status()
                
                # Validate it's an image
                content_type = response.headers.get('content-type', '')
                if 'image' not in content_type.lower():
                    raise ValueError(f"Invalid content type: {content_type}")
                
                # Write image
                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                
                self.stats['successful'] += 1
                return True
                
            except requests.exceptions.Timeout:
                if attempt == self.retries - 1:
                    error_msg = f"Timeout: {url}"
                    self.stats['errors'].append(error_msg)
                continue
            except requests.exceptions.ConnectionError:
                if attempt == self.retries - 1:
                    error_msg = f"Connection error: {url}"
                    self.stats['errors'].append(error_msg)
                continue
            except Exception as e:
                if attempt == self.retries - 1:
                    error_msg = f"Error downloading {url}: {str(e)[:100]}"
                    self.stats['errors'].append(error_msg)
                continue
            
            # Brief delay before retry
            if attempt < self.retries - 1:
                time.sleep(1)
        
        self.stats['failed'] += 1
        return False
    
    def download_images_batch(self, image_links, output_folder, dataset_name="Dataset"):
        """Download multiple images in parallel"""
        os.makedirs(output_folder, exist_ok=True)
        
        # Filter out NaN/None values
        valid_links = [url for url in image_links if url and pd.notna(url)]
        
        print(f"\n{'='*70}")
        print(f"Downloading {dataset_name}: {len(valid_links)} images")
        print(f"{'='*70}")
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {
                executor.submit(self.download_image, url, output_folder): url 
                for url in valid_links
            }
            
            with tqdm(as_completed(futures), total=len(futures), 
                     desc=f"{dataset_name}", unit="image") as pbar:
                for future in pbar:
                    try:
                        future.result()
                    except Exception as e:
                        self.stats['errors'].append(str(e)[:100])
                    pbar.update(1)
        
        # Print statistics
        self.print_stats(dataset_name)
    
    def print_stats(self, dataset_name):
        """Print download statistics"""
        total = (self.stats['successful'] + self.stats['failed'] + 
                self.stats['skipped'])
        print(f"\n{dataset_name} Statistics:")
        print(f"  ✅ Successful: {self.stats['successful']}/{total}")
        print(f"  ⏭️  Skipped (already exist): {self.stats['skipped']}/{total}")
        print(f"  ❌ Failed: {self.stats['failed']}/{total}")
        
        if self.stats['errors']:
            print(f"\n⚠️  Sample Errors (showing first 5):")
            for error in self.stats['errors'][:5]:
                print(f"    - {error}")
    
    def reset_stats(self):
        """Reset statistics for next batch"""
        self.stats = {
            'successful': 0,
            'failed': 0,
            'skipped': 0,
            'errors': []
        }

# ============================================================================
# 5. EXECUTE DOWNLOADS
# ============================================================================
downloader = ImageDownloader(max_workers=4, timeout=20, retries=3)

# Download training images
print("\n" + "="*70)
print("STARTING DOWNLOADS")
print("="*70)

downloader.download_images_batch(
    train_df['image_link'].values,
    TRAIN_IMAGE_FULL_PATH,
    "Training Dataset"
)

# Reset stats for test dataset
downloader.reset_stats()

# Download test images
downloader.download_images_batch(
    test_df['image_link'].values,
    TEST_IMAGE_FULL_PATH,
    "Test Dataset"
)

# ============================================================================
# 6. SUMMARY
# ============================================================================
print("\n" + "="*70)
print("DOWNLOAD COMPLETE")
print("="*70)

train_count = len([f for f in os.listdir(TRAIN_IMAGE_FULL_PATH) 
                   if os.path.isfile(os.path.join(TRAIN_IMAGE_FULL_PATH, f))])
test_count = len([f for f in os.listdir(TEST_IMAGE_FULL_PATH) 
                  if os.path.isfile(os.path.join(TEST_IMAGE_FULL_PATH, f))])

print(f"\n📊 Final Summary:")
print(f"  Training images downloaded: {train_count}")
print(f"  Test images downloaded: {test_count}")
print(f"  Total images: {train_count + test_count}")
print(f"\n✅ Image download is COMPLETE!")
print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ROBUST IMAGE DOWNLOADER
Started at: 2025-10-13 21:38:28

Current Working Directory: C:\Users\Deepak\AMLC\NEW
Base Path: C:\Users\Deepak\AMLC\NEW\AMLC_2025
Data Path: C:\Users\Deepak\AMLC\NEW\AMLC_2025\data
Image Path: C:\Users\Deepak\AMLC\NEW\AMLC_2025\image

Loading datasets...
✅ Train dataset loaded: 75000 rows
✅ Test dataset loaded: 75000 rows

Train images folder: C:\Users\Deepak\AMLC\NEW\AMLC_2025\image\train_full
Test images folder: C:\Users\Deepak\AMLC\NEW\AMLC_2025\image\test_full


STARTING DOWNLOADS



Training Dataset: 100%|███████████████████████████████████████████████████████| 75000/75000 [51:05<00:00, 24.47image/s]



Training Dataset Statistics:
  ✅ Successful: 72287/75000
  ⏭️  Skipped (already exist): 2712/75000
  ❌ Failed: 1/75000

⚠️  Sample Errors (showing first 5):
    - Error downloading https://m.media-amazon.com/images/I/51mjZYDYjyL.jpg: 404 Client Error: Not Found for url: https://m.media-amazon.com/images/I/51mjZYDYjyL.jpg



Test Dataset: 100%|███████████████████████████████████████████████████████████| 75000/75000 [39:25<00:00, 31.71image/s]



Test Dataset Statistics:
  ✅ Successful: 37850/75000
  ⏭️  Skipped (already exist): 2285/75000
  ❌ Failed: 34865/75000

⚠️  Sample Errors (showing first 5):
    - Error downloading https://m.media-amazon.com/images/I/81PfWwQXCkL.jpg: [Errno 28] No space left on device
    - Error downloading https://m.media-amazon.com/images/I/71V0JciZ5qL.jpg: [Errno 28] No space left on device
    - Error downloading https://m.media-amazon.com/images/I/91HYe1PLmSL.jpg: [Errno 28] No space left on device
    - Error downloading https://m.media-amazon.com/images/I/81KQQlXVlAL.jpg: [Errno 28] No space left on device
    - Error downloading https://m.media-amazon.com/images/I/612njcf0TSL.jpg: [Errno 28] No space left on device

DOWNLOAD COMPLETE

📊 Final Summary:
  Training images downloaded: 72287
  Test images downloaded: 38918
  Total images: 111205

✅ Image download is COMPLETE!
Finished at: 2025-10-13 23:09:19
